<a href="https://colab.research.google.com/github/2303A51908/Reinforecement-Learning---B12/blob/main/2303A51908_RL_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.distributions import Categorical

class PolicyNetwork(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(PolicyNetwork, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return F.softmax(x, dim=-1)

def reinforce(env, policy, optimizer, num_episodes=1000, gamma=0.99, print_every=100):
    rewards_history = []
    for episode in range(num_episodes):
        reset_result = env.reset()
        if isinstance(reset_result, tuple):
            state = reset_result[0]
        else:
            state = reset_result

        log_probs = []
        rewards = []
        done = False

        while not done:
            state_tensor = torch.from_numpy(state).float().unsqueeze(0)
            probs = policy(state_tensor)
            m = Categorical(probs)
            action = m.sample()
            log_probs.append(m.log_prob(action))
            step_result = env.step(action.item())
            if len(step_result) == 4:
                state, reward, done, _ = step_result
            else:
                state, reward, terminated, truncated, _ = step_result
                done = terminated or truncated
            rewards.append(reward)

        returns = []
        R = 0
        for r in reversed(rewards):
            R = r + gamma * R
            returns.insert(0, R)

        returns = torch.tensor(returns)
        returns = (returns - returns.mean()) / (returns.std() + 1e-9)

        policy_loss = []
        for log_prob, R in zip(log_probs, returns):
            policy_loss.append(-log_prob * R)
        policy_loss = torch.cat(policy_loss).sum()

        optimizer.zero_grad()
        policy_loss.backward()
        optimizer.step()

        total_reward = sum(rewards)
        rewards_history.append(total_reward)

        if (episode + 1) % print_every == 0:
            print(f"Episode {episode + 1}/{num_episodes}, Average Reward (last {print_every}): "
                  f"{np.mean(rewards_history[-print_every:])}")

    return rewards_history

if __name__ == "__main__":
    env = gym.make('CartPole-v1')
    state_size = env.observation_space.shape[0]
    action_size = env.action_space.n
    hidden_size = 128
    policy = PolicyNetwork(state_size, hidden_size, action_size)
    optimizer = optim.Adam(policy.parameters(), lr=0.01)
    print("Training REINFORCE on CartPole-v1...")
    reinforce(env, policy, optimizer)
    env.close()


Training REINFORCE on CartPole-v1...
Episode 100/1000, Average Reward (last 100): 124.03
Episode 200/1000, Average Reward (last 100): 494.56
Episode 300/1000, Average Reward (last 100): 492.89
Episode 400/1000, Average Reward (last 100): 234.16
Episode 500/1000, Average Reward (last 100): 125.36
Episode 600/1000, Average Reward (last 100): 165.42
Episode 700/1000, Average Reward (last 100): 175.97
Episode 800/1000, Average Reward (last 100): 234.16
Episode 900/1000, Average Reward (last 100): 215.66
Episode 1000/1000, Average Reward (last 100): 224.25
